In [ ]:
import os
import json
import re
import datetime as dt
import pathlib
from typing import Dict, List

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

import pathlib

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Raw-data directory:", DATA_RAW.resolve())


Raw-data directory: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw


## 1. Load the API key from `.env`


In [2]:
load_dotenv()

# The real key is read from the local .env file.
ALPHA_KEY = os.getenv("API_KEY")

print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))


Loaded ALPHAVANTAGE_API_KEY? True


## Helper functions: reproducible filenames and validation

In [3]:
def safe_stamp() -> str:
    return dt.datetime.now().strftime("%Y%m%d-%H%M")


def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join(
        [f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()]
    )
    return f"{prefix}_{mid}_{safe_stamp()}.csv"


def validate_df(
    df: pd.DataFrame,
    required_cols: List[str],
    dtypes_map: Dict[str, str]
) -> Dict:
    """Return a simple validation report without silently changing the data."""
    report = {}

    missing = [c for c in required_cols if c not in df.columns]
    report["missing_cols"] = missing
    report["shape"] = df.shape
    report["na_counts"] = df.isna().sum().to_dict()

    dtype_errors = {}

    for col, expected in dtypes_map.items():
        if col not in df.columns:
            continue

        try:
            if expected == "datetime64[ns]":
                pd.to_datetime(df[col], errors="raise")
            elif expected == "float":
                pd.to_numeric(df[col], errors="raise")
            elif expected == "int":
                pd.to_numeric(df[col], errors="raise")
            elif expected == "text":
                # Basic text validation: values should be strings and non-empty.
                bad = df[col].astype("string").str.strip().eq("").sum()
                if bad:
                    dtype_errors[col] = f"{bad} empty text value(s)"
        except Exception as e:
            dtype_errors[col] = str(e)

    report["dtype_errors"] = dtype_errors
    return report


## 2. API Pull — Alpha Vantage daily TSLA prices

In [6]:
SYMBOL = "TSLA"
API_URL = "https://www.alphavantage.co/query"

api_params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",
    "datatype": "json",
    "apikey": ALPHA_KEY,
}

# ----- Request -----
try:
    response = requests.get(API_URL, params=api_params, timeout=30)
    response.raise_for_status()
    payload = response.json()
except requests.RequestException as e:
    raise RuntimeError(f"Alpha Vantage HTTP request failed: {e}") from e
except ValueError as e:
    raise RuntimeError("Alpha Vantage did not return valid JSON.") from e

# Alpha Vantage can return an explanatory JSON object instead of the time series.
series_key = "Time Series (Daily)"
if series_key not in payload:
    api_message = (
        payload.get("Error Message")
        or payload.get("Note")
        or payload.get("Information")
        or str(payload)[:300]
    )
    raise RuntimeError(f"Alpha Vantage returned no daily time series: {api_message}")

# ----- JSON -> DataFrame -----
df_api = (
    pd.DataFrame.from_dict(payload[series_key], orient="index")
    .rename_axis("date")
    .reset_index()
)

df_api = df_api.rename(columns={
    "1. open": "open",
    "2. high": "high",
    "3. low": "low",
    "4. close": "close",
    "5. volume": "volume",
})

api_required = ["date", "open", "high", "low", "close", "volume"]
df_api = df_api[api_required].copy()

# ----- Parse dtypes -----
df_api["date"] = pd.to_datetime(df_api["date"], errors="raise")

for col in ["open", "high", "low", "close"]:
    df_api[col] = pd.to_numeric(df_api[col], errors="raise").astype(float)

df_api["volume"] = pd.to_numeric(
    df_api["volume"], errors="raise"
).astype("int64")

df_api = df_api.sort_values("date").reset_index(drop=True)

# ----- Validation -----
api_report = validate_df(
    df_api,
    required_cols=api_required,
    dtypes_map={
        "date": "datetime64[ns]",
        "open": "float",
        "high": "float",
        "low": "float",
        "close": "float",
        "volume": "int",
    },
)

print("API validation report:")
print(json.dumps(api_report, indent=2, default=str))

assert api_report["missing_cols"] == [], "Required API columns are missing."
assert df_api.shape[0] > 0, "API DataFrame is empty."
assert df_api[api_required].isna().sum().sum() == 0, "API data contains NA values."
assert df_api["date"].is_unique, "Duplicate dates found."
assert (df_api["high"] >= df_api["low"]).all(), "Found high < low."
assert (df_api["volume"] >= 0).all(), "Found negative volume."

print("\nAPI data preview:")
display(df_api.head())

# ----- Save raw CSV -----
api_filename = safe_filename(
    prefix="api",
    meta={"source": "alphavantage", "ticker": SYMBOL}
)
api_out_path = DATA_RAW / api_filename

df_api.to_csv(api_out_path, index=False)
print("\nSaved API CSV:", api_out_path)


API validation report:
{
  "missing_cols": [],
  "shape": [
    100,
    6
  ],
  "na_counts": {
    "date": 0,
    "open": 0,
    "high": 0,
    "low": 0,
    "close": 0,
    "volume": 0
  },
  "dtype_errors": {}
}

API data preview:


,date,open,high,low,close,volume
0,2026-03-25,389.99,396.23,385.01,385.95,55157265
1,2026-03-26,381.60,384.44,371.87,372.11,55522879
2,2026-03-27,369.69,369.86,359.47,361.83,62065659
3,2026-03-30,365.86,367.29,352.14,355.28,67954405
4,2026-03-31,361.51,373.33,361.00,371.75,75534934



Saved API CSV: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/api_source-alphavantage_ticker-TSLA_20260817-1949.csv


## 3. Scrape a Small Public Table — DJIA annual returns from Wikipedia


In [7]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
headers = {
    "User-Agent": "AFE-Course-Homework/1.0 (educational use)"
}

# ----- Request and parse HTML -----
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
except requests.RequestException as e:
    raise RuntimeError(f"Wikipedia request failed: {e}") from e

soup = BeautifulSoup(resp.text, "html.parser")

# Find the annual-returns table by its semantic header content.
target_table = None

for candidate in soup.find_all("table"):
    first_row = candidate.find("tr")
    if first_row is None:
        continue

    candidate_headers = [
        cell.get_text(" ", strip=True)
        for cell in first_row.find_all(["th", "td"])
    ]
    header_text = " | ".join(candidate_headers)

    if (
        "Year" in header_text
        and "Closing" in header_text
        and "Percentage" in header_text
    ):
        target_table = candidate
        break

if target_table is None:
    raise RuntimeError("Could not find the DJIA annual-returns table.")

# ----- HTML table -> DataFrame -----
rows = []

for tr in target_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in tr.find_all(["th", "td"])
    ]
    if cells:
        rows.append(cells)

if len(rows) < 2:
    raise RuntimeError("The scraped table did not contain data rows.")

raw_header, *raw_data = rows

# Normalize line breaks / repeated whitespace in headers.
clean_header = [re.sub(r"\s+", " ", h).strip() for h in raw_header]

# Keep only rows matching the table width.
raw_data = [row for row in raw_data if len(row) == len(clean_header)]

df_scrape = pd.DataFrame(raw_data, columns=clean_header)

# Normalize expected column names in case whitespace formatting changes.
rename_map = {}
for col in df_scrape.columns:
    col_lower = col.lower()
    if col_lower == "year":
        rename_map[col] = "Year"
    elif "closing" in col_lower and "value" in col_lower:
        rename_map[col] = "Closing value"
    elif "net" in col_lower and "change" in col_lower:
        rename_map[col] = "Net change"
    elif "percentage" in col_lower and "change" in col_lower:
        rename_map[col] = "Percentage change"

df_scrape = df_scrape.rename(columns=rename_map)

scrape_required = [
    "Year",
    "Closing value",
    "Net change",
    "Percentage change",
]

missing_before_parse = [
    c for c in scrape_required if c not in df_scrape.columns
]
if missing_before_parse:
    raise RuntimeError(
        f"Expected scraped columns not found: {missing_before_parse}. "
        f"Found columns: {list(df_scrape.columns)}"
    )

df_scrape = df_scrape[scrape_required].copy()

# ----- Parse numeric columns -----
def clean_number(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("−", "-", regex=False)  # Unicode minus -> ASCII minus
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


df_scrape["Year"] = clean_number(df_scrape["Year"]).astype("Int64")
df_scrape["Closing value"] = clean_number(df_scrape["Closing value"])
df_scrape["Net change"] = clean_number(df_scrape["Net change"])
df_scrape["Percentage change"] = clean_number(df_scrape["Percentage change"])

# ----- Validation -----
scrape_report = validate_df(
    df_scrape,
    required_cols=scrape_required,
    dtypes_map={
        "Year": "int",
        "Closing value": "float",
        "Net change": "float",
        "Percentage change": "float",
    },
)

print("Scrape validation report:")
print(json.dumps(scrape_report, indent=2, default=str))

assert scrape_report["missing_cols"] == [], "Required scraped columns are missing."
assert df_scrape.shape[0] > 0, "Scraped DataFrame is empty."
assert df_scrape[scrape_required].isna().sum().sum() == 0, (
    "Scraped numeric conversion produced NA values."
)
assert df_scrape["Year"].is_unique, "Duplicate years found."
assert df_scrape["Year"].between(1800, 2100).all(), "Implausible year found."

print("\nScraped data preview:")
display(df_scrape.tail())

# ----- Save raw CSV -----
scrape_filename = safe_filename(
    prefix="scrape",
    meta={"site": "wikipedia", "table": "djia-annual-returns"}
)
scrape_out_path = DATA_RAW / scrape_filename

df_scrape.to_csv(scrape_out_path, index=False)
print("\nSaved scraped CSV:", scrape_out_path)


Scrape validation report:
{
  "missing_cols": [],
  "shape": [
    130,
    4
  ],
  "na_counts": {
    "Year": 0,
    "Closing value": 0,
    "Net change": 0,
    "Percentage change": 0
  },
  "dtype_errors": {}
}

Scraped data preview:


,Year,Closing value,Net change,Percentage change
125,2021,36338.30,5731.82,18.73
126,2022,33147.25,-3191.05,-8.78
127,2023,37689.54,4542.29,13.70
128,2024,42544.22,4854.68,12.88
129,2025,48063.29,5519.07,12.97



Saved scraped CSV: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/scrape_site-wikipedia_table-djia-annual-returns_20260817-1952.csv


## 4. Documentation

### Data sources and parameters

**API dataset**

- Source: Alpha Vantage
- URL: `https://www.alphavantage.co/query`
- Endpoint/function: `TIME_SERIES_DAILY`
- Ticker: `TSLA`
- `outputsize=compact`
- `datatype=json`
- API key: loaded locally from `.env`

**Scraped dataset**

- Source: Wikipedia
- URL: `https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average`
- Table: DJIA annual returns
- Parser: `requests` + `BeautifulSoup`

### Validation logic

For the API dataset:

- require `date`, `open`, `high`, `low`, `close`, and `volume`;
- report DataFrame shape and NA counts;
- parse `date` as datetime;
- parse OHLC as floating-point numbers;
- parse `volume` as an integer;
- require non-empty data;
- require unique dates;
- require `high >= low`;
- require non-negative volume.

For the scraped table:

- require `Year`, `Closing value`, `Net change`, and `Percentage change`;
- report DataFrame shape and NA counts;
- convert all fields to numeric values after cleaning formatting characters;
- require non-empty data;
- require unique, plausible years.



### Assumptions & risks

- Alpha Vantage may return rate-limit or service messages instead of a time-series payload; the notebook checks for the expected `Time Series (Daily)` key and raises a clear error.
- The API requires internet access and a valid API key.
- Website HTML can change. The scraping code searches for table header content rather than relying only on table position, but a substantial Wikipedia layout or header change could still require an update.
- Scraped data is treated as source data; this notebook does not verify Wikipedia values against an independent market-data provider.
- Re-running the notebook creates timestamped files rather than silently overwriting earlier raw data.


In [8]:
# Final check: show the raw files created during this run.
print("API file:", api_out_path)
print("Scrape file:", scrape_out_path)

print("\nFiles currently in data/raw/:")
for path in sorted(DATA_RAW.glob("*.csv")):
    print(" -", path)


API file: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/api_source-alphavantage_ticker-TSLA_20260817-1949.csv
Scrape file: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/scrape_site-wikipedia_table-djia-annual-returns_20260817-1952.csv

Files currently in data/raw/:
 - /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/api_source-alphavantage_ticker-TSLA_20260817-1949.csv
 - /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework4/data/raw/scrape_site-wikipedia_table-djia-annual-returns_20260817-1952.csv
